In [ ]:
import sys

sys.path.append("..")

import torch
import torchmetrics
from lightning import Trainer, seed_everything

from src import LightningDataset, Module
from src.constants import DEFAULT_SEED
from src.datasets import CustomDataset
from src.transforms import LineGraph

In [ ]:
CKPT = "../lightning_logs/version_45927073/checkpoints/epoch=101-step=131378.ckpt"
BATCH_SIZE = 1

In [ ]:
_ = seed_everything(DEFAULT_SEED, verbose=False)

In [ ]:
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
k = ckpt["datamodule_hyper_parameters"]["k"]

In [ ]:
from pprint import pprint

ckpt_params = ckpt["hyper_parameters"] | ckpt["datamodule_hyper_parameters"]
pprint(ckpt_params)

In [ ]:
datamodule = LightningDataset(
    pred_dataset=CustomDataset(pre_transform=LineGraph(), k=k),
    # pred_dataset=CustomDataset(root="../data/test", pre_transform=LineGraph(), k=k),
    num_workers=4,
    batch_size=BATCH_SIZE,
    k=ckpt_params["k"],
)

In [ ]:
metrics = torchmetrics.MetricCollection(
    {
        "f1": torchmetrics.F1Score(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "auroc": torchmetrics.AUROC(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "acc": torchmetrics.Accuracy(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "confmat": torchmetrics.ConfusionMatrix(
            task="multiclass", num_classes=ckpt_params["num_classes"]
        ),
    }
)

In [ ]:
model = Module.load_from_checkpoint(checkpoint_path=CKPT, metrics=metrics, weights_only=False)

In [ ]:
trainer = Trainer(
    precision="16-mixed" if torch.cuda.is_available() else 32,
    deterministic=False,
    enable_progress_bar=True,
)

In [ ]:
predictions = trainer.predict(model=model, datamodule=datamodule)